# Topic 2: Data Preprocessing & Feature Engineering

This notebook is about preprocessing. As far as steps go, we will perform the following in this notebook:
1. Handle DAYS_EMPLOYED sentinel value + identify any other sentinel values in the dataset
2. Identify features with >40\% missing data, and decide whether to drop, impute, or keep with indicator (document rationale)
3. Create missingness indicators for all features with >5% missing. Test whether missingness is predictive of TARGET (fit a logistic regression on indicators only; report AUROC). 
4. Impute remaining numeric features with median; categorical features with mode. 
------------------------------------------------------- 

In [59]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [60]:
# import dataset
application_train_path = Path(f"{os.getcwd()}/../data/application_train.csv")
application_train = pd.read_csv(application_train_path)

application_test_path = Path(f"{os.getcwd()}/../data/application_test.csv")
application_test = pd.read_csv(application_test_path)

### 1.a. Replace DAYS_EMPLOYED sentinel with NaN

In [61]:
application_train['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

In [62]:
application_train['DAYS_EMPLOYED'].describe()

count    252137.000000
mean      -2384.169325
std        2338.360162
min      -17912.000000
25%       -3175.000000
50%       -1648.000000
75%        -767.000000
max           0.000000
Name: DAYS_EMPLOYED, dtype: float64

### 1.b. Identify any other sentinel values

In [63]:
application_train.describe()

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
count,307511.000000,307511.000000,307511.000000,3.075110e+05,3.075110e+05,307499.000000,3.072330e+05,307511.000000,307511.000000,252137.000000,...,307511.000000,307511.000000,307511.000000,307511.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000,265992.000000
mean,278180.518577,0.080729,0.417052,1.687979e+05,5.990260e+05,27108.573909,5.383962e+05,0.020868,-16036.995067,-2384.169325,...,0.008130,0.000595,0.000507,0.000335,0.006402,0.007000,0.034362,0.267395,0.265474,1.899974
std,102790.175348,0.272419,0.722121,2.371231e+05,4.024908e+05,14493.737315,3.694465e+05,0.013831,4363.988632,2338.360162,...,0.089798,0.024387,0.022518,0.018299,0.083849,0.110757,0.204685,0.916002,0.794056,1.869295
min,100002.000000,0.000000,0.000000,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,0.000290,-25229.000000,-17912.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,189145.500000,0.000000,0.000000,1.125000e+05,2.700000e+05,16524.000000,2.385000e+05,0.010006,-19682.000000,-3175.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,278202.000000,0.000000,0.000000,1.471500e+05,5.135310e+05,24903.000000,4.500000e+05,0.018850,-15750.000000,-1648.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,367142.500000,0.000000,1.000000,2.025000e+05,8.086500e+05,34596.000000,6.795000e+05,0.028663,-12413.000000,-767.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
max,456255.000000,1.000000,19.000000,1.170000e+08,4.050000e+06,258025.500000,4.050000e+06,0.072508,-7489.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,4.000000,9.000000,8.000000,27.000000,261.000000,25.000000


### 2. Decide whether to keep/drop/impute for features with >40\% missing

We drop features with more than 40\% missingness, these won't be useful. Deciding to impute/keep these would just introduce unnecessary noise

In [64]:
def drop_more_than_40_pct_missing():
    pct_na = pd.DataFrame(application_train.isna().mean()*100).rename(columns={0:'pct_na'})
    # get columns with more than 40% missing values
    over_40pct_missing = pct_na[pct_na['pct_na'] > 40.0].index
    application_train.dropna(subset=over_40pct_missing, axis=0, inplace=True)

drop_more_than_40_pct_missing()

In [65]:
# now get percentage of missing values then sort by descending to see if we got rid of all rows with more than 40% missing 
application_train.isna().mean().sort_values(ascending=False)*100

OCCUPATION_TYPE                21.206046
EXT_SOURCE_3                   15.314621
AMT_REQ_CREDIT_BUREAU_YEAR      9.762492
AMT_REQ_CREDIT_BUREAU_QRT       9.762492
AMT_REQ_CREDIT_BUREAU_MON       9.762492
                                 ...    
REG_CITY_NOT_WORK_CITY          0.000000
REG_CITY_NOT_LIVE_CITY          0.000000
LIVE_REGION_NOT_WORK_REGION     0.000000
REG_REGION_NOT_WORK_REGION      0.000000
YEARS_BUILD_MODE                0.000000
Length: 122, dtype: float64

### 3. Create missingness indicators for features with >5\% missing

In [66]:
# get columns with less than 40% but more than 5% missing values
# since we dropped some rows containing nulls, this will affect the missing percentages for other features
# recompute, get features with more than 5% missing

indicators = []

# function to create missingness indicators for all features with more than 5% missing values
def create_missingness_indicators():
    pct_na = pd.DataFrame(application_train.isna().mean()*100).rename(columns={0:'pct_na'})
    over_5pct_missing_filter = pct_na['pct_na'] > 5.0
    over_5pct_missing = pct_na[over_5pct_missing_filter]
    display(over_5pct_missing)
    print(f"{len(over_5pct_missing)} features with >5% missingness")
    for v in over_5pct_missing.index:
        indicator(v)

# function to create a new column with missingness indicator
def indicator(col):
    col_name = f"missingness_{col}"
    # isna returns a boolean series by default, we convert it to int (0/1) so that we can test correlation with TARGET
    application_train[col_name] = application_train[col].isna().astype(int)
    indicators.append(col_name)

create_missingness_indicators()
display(application_train[indicators])

,pct_na
DAYS_EMPLOYED,5.313078
OCCUPATION_TYPE,21.206046
EXT_SOURCE_3,15.314621
AMT_REQ_CREDIT_BUREAU_HOUR,9.762492
AMT_REQ_CREDIT_BUREAU_DAY,9.762492
AMT_REQ_CREDIT_BUREAU_WEEK,9.762492
AMT_REQ_CREDIT_BUREAU_MON,9.762492
AMT_REQ_CREDIT_BUREAU_QRT,9.762492
AMT_REQ_CREDIT_BUREAU_YEAR,9.762492


9 features with >5% missingness


,missingness_DAYS_EMPLOYED,missingness_OCCUPATION_TYPE,missingness_EXT_SOURCE_3,missingness_AMT_REQ_CREDIT_BUREAU_HOUR,missingness_AMT_REQ_CREDIT_BUREAU_DAY,missingness_AMT_REQ_CREDIT_BUREAU_WEEK,missingness_AMT_REQ_CREDIT_BUREAU_MON,missingness_AMT_REQ_CREDIT_BUREAU_QRT,missingness_AMT_REQ_CREDIT_BUREAU_YEAR
71,0,0,0,0,0,0,0,0,0
124,0,0,0,0,0,0,0,0,0
143,1,1,0,0,0,0,0,0,0
152,0,0,0,0,0,0,0,0,0
161,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
307407,0,0,0,0,0,0,0,0,0
307456,0,0,0,0,0,0,0,0,0
307459,0,0,1,1,1,1,1,1,1
307474,0,0,1,1,1,1,1,1,1


In [67]:
display(application_train)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,AMT_REQ_CREDIT_BUREAU_YEAR,missingness_DAYS_EMPLOYED,missingness_OCCUPATION_TYPE,missingness_EXT_SOURCE_3,missingness_AMT_REQ_CREDIT_BUREAU_HOUR,missingness_AMT_REQ_CREDIT_BUREAU_DAY,missingness_AMT_REQ_CREDIT_BUREAU_WEEK,missingness_AMT_REQ_CREDIT_BUREAU_MON,missingness_AMT_REQ_CREDIT_BUREAU_QRT,missingness_AMT_REQ_CREDIT_BUREAU_YEAR
71,100083,0,Cash loans,M,Y,Y,0,103500.0,573628.5,24435.0,...,3.0,0,0,0,0,0,0,0,0,0
124,100145,0,Cash loans,F,Y,Y,1,202500.0,260725.5,16789.5,...,3.0,0,0,0,0,0,0,0,0,0
143,100165,0,Cash loans,F,Y,Y,0,175500.0,1293502.5,35568.0,...,2.0,1,1,0,0,0,0,0,0,0
152,100179,0,Cash loans,F,Y,N,0,202500.0,675000.0,53329.5,...,4.0,0,0,0,0,0,0,0,0,0
161,100190,0,Cash loans,M,Y,N,0,162000.0,263686.5,24781.5,...,3.0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307407,456140,1,Cash loans,F,Y,Y,1,261000.0,711454.5,47673.0,...,1.0,0,0,0,0,0,0,0,0,0
307456,456195,0,Cash loans,F,Y,Y,0,94500.0,270000.0,15075.0,...,3.0,0,0,0,0,0,0,0,0,0
307459,456198,0,Cash loans,M,Y,Y,0,225000.0,959017.5,49095.0,...,NaN,0,0,1,1,1,1,1,1,1
307474,456214,0,Cash loans,M,Y,Y,2,135000.0,360000.0,23004.0,...,NaN,0,0,1,1,1,1,1,1,1


In [68]:
# fit logistic regression on only indicators
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def test_missingness_predictability():
    lr = LogisticRegression()
    # create "testing" set from a subset of training dataset

    X = application_train.loc[:, application_train.columns.isin(indicators)]
    y = application_train['TARGET']

    # split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # fit
    lr.fit(X_train, y_train)
    preds = lr.predict(X_test)

    # metrics
    auroc = roc_auc_score(y_test, preds)
    print(f"AUROC is: {auroc}")

test_missingness_predictability()

AUROC is: 0.5


Based on the above, it would seem that missingness has close to no correlation with/is not a a meaningful predictor of the target. In other words, one would not be inclined to use the missingness features as a reliable predictor of whether an application would be defaulted. To back this, our AUROC value of 0.5 indicates that the model is no better at distinguishing between the two classes than a random guess.

### 4. For remaining features, impute numerical using median, categorical using mode

In [69]:
def impute_remaining():
    pct_na = pd.DataFrame(application_train.isna().mean()*100).rename(columns={0:'pct_na'})

    # find columns that have less than 5pct but nonzero (have not yet been filled)
    less_than_5pct_filter = (pct_na['pct_na'] <= 5.0) & (pct_na['pct_na'] != 0.0)
    less_than_5pct = pct_na[less_than_5pct_filter].sort_values(by=['pct_na'], ascending=False)
    less_than_5pct_df = application_train[less_than_5pct.index]
                                          
    # split into numeric and categorical, impute differently
    numeric_columns = less_than_5pct_df.select_dtypes(include=['number']).columns
    categorical_columns = less_than_5pct_df.select_dtypes(include=['object']).columns
    
    # fill categorical columns with mode
    application_train[categorical_columns] = application_train[categorical_columns].fillna(value=application_train[categorical_columns].mode().iloc[0])

    # fill numeric columns with median
    application_train[numeric_columns] = application_train[numeric_columns].fillna(value=application_train[numeric_columns].median())

impute_remaining()

In [70]:
pct_na = pd.DataFrame(application_train.isna().mean()*100).rename(columns={0:'pct_na'})
display(pct_na.value_counts())

pct_na   
0.000000     122
9.762492       6
5.313078       1
15.314621      1
21.206046      1
Name: count, dtype: int64

The 9 columns that have nonzero missing value percentages are the ones from which we created missingness indicators (usually we keep the original column)

### Save the file in for future use

In [72]:
clean_data_filepath = Path(f"{os.getcwd()}/../data/application_train_cleaned.csv")
application_train.to_csv(clean_data_filepath)